# CHALLENGE SPRINT 4 - MACHINE LEARNING & MODELOS LINEARES



---
# 01) Análise do Dataset (3,0 pontos)

### a) Carregamento do Dataset em um DataFrame

# Baixando Bibliotecas necessárias

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

In [ ]:

# a) Carregamento dos dados
df = pd.read_csv('Renewable_Energy_Data.csv')

print(f"Total de Linhas: {df.shape[0]} | Total de Colunas: {df.shape[1]}")
display(df.head())
print("\nInformações gerais do DataFrame:")
print(df.info())

### b) Encoding da Variável-Alvo (`Energy_Class`)

A variável de interesse `Energy_Class` possui natureza categórica ordinal. Realizamos a codificação para valores inteiros:
- `Low` $\rightarrow 0$
- `Medium` $\rightarrow 1$
- `High` $\rightarrow 2$

In [ ]:
# b) Encoding ordinal da variável-alvo
target_mapping = {'Low': 0, 'Medium': 1, 'High': 2}
df['Energy_Class_Encoded'] = df['Energy_Class'].map(target_mapping)

print("Frequência absoluta por classe (Original vs Encoded):")
dist_df = pd.DataFrame({
    'Classe Original': df['Energy_Class'].value_counts().index,
    'Valor Numérico': [target_mapping[k] for k in df['Energy_Class'].value_counts().index],
    'Quantidade': df['Energy_Class'].value_counts().values,
    'Proporção (%)': (df['Energy_Class'].value_counts(normalize=True).values * 100).round(2)
})
display(dist_df)

### c) Matriz de Correlação Linear entre as Features e a Variável-Alvo

In [ ]:
# c) Matriz de correlação de Pearson
numerical_cols = df.select_dtypes(include=['float64', 'int64']).columns
corr_matrix = df[numerical_cols].corr()

print("Matriz de Correlação de Pearson:")
display(corr_matrix.round(4))

# Gráfico Heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt='.3f', vmin=-1, vmax=1, linewidths=0.5)
plt.title('Matriz de Correlação das Features com Energy_Class_Encoded', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# Correlação isolada com a variável-alvo ordenada por magnitude
corr_target = corr_matrix['Energy_Class_Encoded'].drop('Energy_Class_Encoded').sort_values(ascending=False)
print("\nCorrelação de cada feature com Energy_Class_Encoded:")
display(corr_target.to_frame(name='Correlação de Pearson (r)'))

### d) Avaliação e Justificativa Técnica sobre Seleção de Features

**Análise da Matriz de Correlação e Decisão Técnica:**

1. **Análise de Multicolinearidade:**
   - Entre as variáveis preditoras (`Temperature_C`, `Wind_Speed_m_s`, `Solar_Radiation_kWh_m2`, `Rainfall_mm`, `Efficiency_Ratio`, `Lagged_Production_MWh`, `Combined_Weather_Index`), a correlação entre quaisquer pares é baixa ($|r| < 0.30$). O maior valor absoluto encontrado é entre `Efficiency_Ratio` e `Temperature_C` ($r = -0.260$). Conclui-se que **não há multicolinearidade** ou redundância informacional entre os atributos preditores.

2. **Poder Preditivo Multivariado:**
   - A variável `Efficiency_Ratio` exibe correlação linear positiva relevante com a classe de energia ($r = +0.211$).
   - As demais variáveis climáticas/históricas possuem correlações univariadas pequenas ($|r| < 0.05$). No entanto, modelos lineares multivariados constroem uma combinação afim $z = \mathbf{w}^T \mathbf{x} + b$, na qual múltiplas variáveis climáticas fracamente correlacionadas de forma isolada somam poder discriminativo conjunto no espaço $\mathbb{R}^n$.

3. **Decisão:**
   - Optou-se por **manter todas as 7 features numéricas contínuas** no modelo, acompanhadas de **padronização de escala (`StandardScaler`)**.
   - **Justificativa:** Como a dimensionalidade é pequena ($n=7$) em relação ao número de observações ($N=1000$), não há risco de sobreajuste por alta dimensionalidade (*curse of dimensionality*). A inclusão de todos os dados físicos de produção renovável permite que o classificador linear pondere de forma matemática a importância real de cada grandeza física.

---
# 02) Definição e Aplicação do Modelo (4,0 pontos)

### a) Modelo de Classificação Linear Selecionado
O modelo selecionado é a **Regressão Logística Multinomial (`sklearn.linear_model.LogisticRegression`)**.

### b) Por que o modelo é considerado Linear e sua Fronteira de Decisão (*Decision Boundary*)

A **Regressão Logística** é classificada teoricamente como um **modelo linear** porque sua fronteira de decisão (*decision boundary*) que particiona o espaço de características é um **hiperplano afim (linear)**.

**Demonstração Matemática:**
1. Para cada classe $k \in \{0, 1, 2\}$, o modelo calcula uma combinação linear ponderada das variáveis de entrada $\mathbf{x} = [x_1, x_2, \dots, x_n]^T$:
   $$z_k = \mathbf{w}_k^T \mathbf{x} + b_k = \sum_{i=1}^{n} w_{k,i} x_i + b_k$$
2. A probabilidade de pertinência à classe $k$ é obtida através da função Softmax:
   $$P(Y = k \mid \mathbf{x}) = \frac{e^{z_k}}{\sum_{j} e^{z_j}}$$
3. A fronteira de decisão entre duas classes arbitrárias $k$ e $j$ é o conjunto de pontos $\mathbf{x}$ onde as probabilidades estimadas são exatamente iguais ($P(Y=k\mid\mathbf{x}) = P(Y=j\mid\mathbf{x})$):
   $$z_k = z_j \iff \mathbf{w}_k^T \mathbf{x} + b_k = \mathbf{w}_j^T \mathbf{x} + b_j \iff (\mathbf{w}_k - \mathbf{w}_j)^T \mathbf{x} + (b_k - b_j) = 0$$
4. A equação resultante $(\mathbf{w}_k - \mathbf{w}_j)^T \mathbf{x} + (b_k - b_j) = 0$ é estritamente a equação de um **hiperplano linear** em $\mathbb{R}^n$ (uma reta em 2D, um plano em 3D e um hiperplano afim em $n$ dimensões). Portanto, a separação entre classes é 100% linear.

### c) e d) Treinamento do Modelo nos Cenários 1 e 2

- **Cenário 1:** 60% dos dados para treinamento e 40% para teste.
- **Cenário 2:** 85% dos dados para treinamento e 15% para teste.
- **Condições padronizadas:** `random_state=42`, amostragem estratificada (`stratify=y`) e padronização `StandardScaler` aplicada rigorosamente em ambos.

In [ ]:


# Definição das variáveis de entrada e saída
features = ['Temperature_C', 'Wind_Speed_m_s', 'Solar_Radiation_kWh_m2', 
            'Rainfall_mm', 'Efficiency_Ratio', 'Lagged_Production_MWh', 'Combined_Weather_Index']
X = df[features]
y = df['Energy_Class_Encoded']
class_names = ['Low', 'Medium', 'High']

# =====================================================================
# CENÁRIO 1: 60% Treino / 40% Teste
# =====================================================================
X_train_c1, X_test_c1, y_train_c1, y_test_c1 = train_test_split(
    X, y, test_size=0.40, random_state=42, stratify=y
)
scaler_c1 = StandardScaler()
X_train_c1_s = scaler_c1.fit_transform(X_train_c1)
X_test_c1_s = scaler_c1.transform(X_test_c1)

model_c1 = LogisticRegression(random_state=42, max_iter=1000)
model_c1.fit(X_train_c1_s, y_train_c1)
y_pred_c1 = model_c1.predict(X_test_c1_s)

# =====================================================================
# CENÁRIO 2: 85% Treino / 15% Teste
# =====================================================================
X_train_c2, X_test_c2, y_train_c2, y_test_c2 = train_test_split(
    X, y, test_size=0.15, random_state=42, stratify=y
)
scaler_c2 = StandardScaler()
X_train_c2_s = scaler_c2.fit_transform(X_train_c2)
X_test_c2_s = scaler_c2.transform(X_test_c2)

model_c2 = LogisticRegression(random_state=42, max_iter=1000)
model_c2.fit(X_train_c2_s, y_train_c2)
y_pred_c2 = model_c2.predict(X_test_c2_s)

print(f"Cenário 1 treinado com sucesso! Amostras: {len(X_train_c1)} treino / {len(X_test_c1)} teste.")
print(f"Cenário 2 treinado com sucesso! Amostras: {len(X_train_c2)} treino / {len(X_test_c2)} teste.")

---
# 03) Avaliação da Performance (3,0 pontos)

### a) Matriz de Confusão para Cada Cenário

In [ ]:

cm_c1 = confusion_matrix(y_test_c1, y_pred_c1)
cm_c2 = confusion_matrix(y_test_c2, y_pred_c2)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Heatmap Cenário 1
disp_c1 = ConfusionMatrixDisplay(confusion_matrix=cm_c1, display_labels=class_names)
disp_c1.plot(cmap='Blues', ax=axes[0], values_format='d')
axes[0].set_title('Matriz de Confusão - Cenário 1 (60/40)', fontsize=12, fontweight='bold')

# Heatmap Cenário 2
disp_c2 = ConfusionMatrixDisplay(confusion_matrix=cm_c2, display_labels=class_names)
disp_c2.plot(cmap='Greens', ax=axes[1], values_format='d')
axes[1].set_title('Matriz de Confusão - Cenário 2 (85/15)', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

### b) Cálculo de Acurácia, Precisão e Recall para Cada Cenário

In [ ]:

# Métricas Globais Cenário 1
acc_c1 = accuracy_score(y_test_c1, y_pred_c1)
prec_c1_macro = precision_score(y_test_c1, y_pred_c1, average='macro')
rec_c1_macro = recall_score(y_test_c1, y_pred_c1, average='macro')
prec_c1_class = precision_score(y_test_c1, y_pred_c1, average=None)
rec_c1_class = recall_score(y_test_c1, y_pred_c1, average=None)

# Métricas Globais Cenário 2
acc_c2 = accuracy_score(y_test_c2, y_pred_c2)
prec_c2_macro = precision_score(y_test_c2, y_pred_c2, average='macro')
rec_c2_macro = recall_score(y_test_c2, y_pred_c2, average='macro')
prec_c2_class = precision_score(y_test_c2, y_pred_c2, average=None)
rec_c2_class = recall_score(y_test_c2, y_pred_c2, average=None)

# Tabela Resumo Consolidada
metrics_table = pd.DataFrame({
    'Métrica / Classe': [
        'Acurácia Global',
        'Precisão (Macro)',
        'Recall (Macro)',
        'Precisão - Low (0)',
        'Recall - Low (0)',
        'Precisão - Medium (1)',
        'Recall - Medium (1)',
        'Precisão - High (2)',
        'Recall - High (2)'
    ],
    'Cenário 1 (60% Treino / 40% Teste)': [
        f'{acc_c1:.4f} ({acc_c1*100:.2f}%)',
        f'{prec_c1_macro:.4f}',
        f'{rec_c1_macro:.4f}',
        f'{prec_c1_class[0]:.4f}',
        f'{rec_c1_class[0]:.4f}',
        f'{prec_c1_class[1]:.4f}',
        f'{rec_c1_class[1]:.4f}',
        f'{prec_c1_class[2]:.4f}',
        f'{rec_c1_class[2]:.4f}'
    ],
    'Cenário 2 (85% Treino / 15% Teste)': [
        f'{acc_c2:.4f} ({acc_c2*100:.2f}%)',
        f'{prec_c2_macro:.4f}',
        f'{rec_c2_macro:.4f}',
        f'{prec_c2_class[0]:.4f}',
        f'{rec_c2_class[0]:.4f}',
        f'{prec_c2_class[1]:.4f}',
        f'{rec_c2_class[1]:.4f}',
        f'{prec_c2_class[2]:.4f}',
        f'{rec_c2_class[2]:.4f}'
    ]
})

print("Tabela Consolidada de Métricas de Performance:")
display(metrics_table)

print("\n--- Relatório Detalhado: Cenário 1 ---")
print(classification_report(y_test_c1, y_pred_c1, target_names=class_names, digits=4))
print("\n--- Relatório Detalhado: Cenário 2 ---")
print(classification_report(y_test_c2, y_pred_c2, target_names=class_names, digits=4))

### c) Análise Crítica dos Resultados e Adequação do Desempenho

**O desempenho obtido é adequado para o problema?**

1. **Acurácia Global Satisfatória (~78% - 79%):**
   - Em um problema multiclasse com 3 categorias (`Low`, `Medium`, `High`), um baseline puramente aleatório teria acurácia de ~33%, e a predição da classe majoritária (`High`) obteria ~48,6%. O modelo linear alcançou **79,00% de acurácia**, comprovando aprendizado real dos padrões de produção de energia.

2. **Excelente Identificação da Classe Majoritária (`High`):**
   - O recall para a classe `High` atingiu **96,92% no Cenário 1** e **94,52% no Cenário 2**, com precisão superior a **81%**. Isso indica que momentos de alta geração de energia renovável são capturados com altíssima confiabilidade e pouquíssimos falsos negativos.

3. **Desafio nas Classes Intermediária (`Medium`) e Minoritária (`Low`):**
   - A classe `Medium` apresenta alguma confusão com as classes adjacentes (`Low` e `High`), o que é esperado devido à continuidade física das grandezas meteorológicas e do `Efficiency_Ratio` nas fronteiras entre faixas.
   - A classe `Low` apresentou alta precisão (92,5% no Cenário 1), com recall mais modesto (52,1% no Cenário 1, subindo para 63,0% no Cenário 2 quando mais dados de treino foram disponibilizados).

**Conclusão:** O desempenho é considerado **bastante adequado e consistente** para um classificador estritamente linear, atendendo com robustez às características físicas do dataset.

### d) Comparação entre os Resultados (40% de Teste vs 15% de Teste)

| Aspecto de Comparação | Cenário 1 (40% Teste) | Cenário 2 (15% Teste) | Análise Comparativa |
| :--- | :---: | :---: | :--- |
| **Acurácia Global** | **79,00%** | **78,00%** | Diferença marginal de apenas 1,0 ponto percentual, evidenciando estabilidade. |
| **Amostras de Teste** | 400 | 150 | Cenário 1 possui quase o triplo de instâncias de teste. |
| **Recall da Classe `Low`** | 52,11% | **62,96%** | Aumento de **+10,85%** no Cenário 2 devido ao maior volume de treino (850 amostras). |
| **Precisão da Classe `Low`** | **92,50%** | 80,95% | Cenário 1 com precisão mais estrita; Cenário 2 mais balanceado. |
| **F1-Score Macro** | 0,7464 | 0,7467 | Praticamente idêntico entre os dois cenários. |

**Síntese Comparativa:** Ambos os cenários apresentaram eficácia comparável. O aumento do conjunto de treino no Cenário 2 (de 600 para 850 amostras) beneficiou o aprendizado da classe minoritária (`Low`), elevando significativamente sua taxa de acerto (*Recall*).

### e) Vantagens e Limitações do Tamanho do Conjunto de Teste

A escolha da divisão treino/teste envolve um **trade-off fundamental entre capacidade de aprendizado e confiabilidade estatística da avaliação**:

#### 1. Conjunto de Teste Maior (ex.: 40% de Teste / Cenário 1):
- **Vantagens:**
  - **Alta Confiabilidade Estatística:** Com 400 instâncias de teste, a estimativa do erro de generalização possui menor variância amostral e intervalos de confiança mais estreitos.
  - **Detecção de Fraquezas:** Expõe o modelo a uma diversidade maior de cenários não vistos, tornando a avaliação mais rigorosa.
- **Limitações:**
  - **Menor Volume de Treinamento:** Ao reservar 40% para teste, restam apenas 60% (600 amostras) para o ajuste dos coeficientes, o que pode prejudicar o aprendizado de classes minoritárias (como observado no menor recall da classe `Low`).

#### 2. Conjunto de Teste Menor (ex.: 15% de Teste / Cenário 2):
- **Vantagens:**
  - **Maior Capacidade de Aprendizado:** O modelo dispõe de 85% dos dados (850 amostras) para ajustar seus hiperplanos, aprendendo fronteiras mais precisas para classes raras.
- **Limitações:**
  - **Menor Representatividade no Teste:** Com apenas 150 instâncias (e apenas 27 da classe `Low`), cada erro individual causa uma variação percentual maior nas métricas (maior variância na estimativa do desempenho real).